<div style="text-align: center;">
    <a href="https://www.hi-paris.fr/">
        <img border="0" src="https://www.hi-paris.fr/wp-content/uploads/2020/09/logo-hi-paris-retina.png" width="25%"></a>
    <a href="https://www.dataia.eu/">
        <img border="0" src="https://github.com/ramp-kits/template-kit/raw/main/img/DATAIA-h.png" width="70%"></a>
</div>

# EEG Alpha Waves Classification Challenge

This challenge is based on the EEG Alpha Waves dataset (Rodrigues2017), originally released by GIPSA-lab and hosted on Zenodo:

https://zenodo.org/record/2348892

The goal is to build a binary classifier from EEG-derived features.

## Dataset description

The dataset contains EEG recordings from multiple subjects.

During data preparation (`tools/setup_data.py`):

- The dataset is downloaded automatically via MOABB / MNE
- EEG signals are segmented into 2-second windows
- For each channel and window, we compute:
  - Mean
  - Standard deviation
  - Alpha band power (8–12 Hz)

This produces a tabular dataset usable with standard scikit-learn models.

The binary labels are constructed using:
- Event annotations when available
- Otherwise a fallback time-based split (early vs late recording)

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ingestion_program.ingestion import get_train_data

X_df, y_df = get_train_data("dev_phase/input_data")
y = y_df.values.ravel()

print("Shape of X:", X_df.shape)
print("Shape of y:", y.shape)

In [ ]:
print("Class distribution:")
print(pd.Series(y).value_counts(normalize=True))

In [ ]:
plt.hist(X_df.iloc[:, 0], bins=30)
plt.title("Distribution of first feature")
plt.show()

## Evaluation strategy

### Metric

We use **Accuracy** as the evaluation metric.

Accuracy is defined as:

Accuracy = (Number of correct predictions) / (Total number of predictions)

Since the classes are balanced (~50/50), accuracy is appropriate and interpretable.

### Splitting strategy

We use a **subject-wise split**:

- Train
- Public test
- Private test

This prevents data leakage across subjects and better reflects generalization to unseen participants.

The public test score is shown on the leaderboard.
The private test score determines final ranking.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

model = LogisticRegression(max_iter=200)

model.fit(X_df, y)

X_test = pd.read_csv("dev_phase/input_data/test/test_features.csv")
y_test = pd.read_csv("dev_phase/reference_data/test_labels.csv").values.ravel()

pred = model.predict(X_test)

print("Baseline accuracy on public test:", accuracy_score(y_test, pred))

## Local testing pipeline

Submissions must provide a `submission.py` file with a function:

```python
def get_model():
    return <sklearn_model>

In [ ]:
from scoring_program.scoring import compute_accuracy

y_pred_df = pd.DataFrame(pred)
print("Accuracy using challenge scoring:", compute_accuracy(y_pred_df, pd.DataFrame(y_test)))